In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from datetime import date
import re
from tqdm import tqdm
import time
from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr
from scipy.spatial.distance import jaccard, euclidean, cosine
import functools
import operator
import gzip

today = date.today()

In [2]:
# 25kb bins
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)
print(bins_25kb)


# SNP data
snp_cols = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'pvalue', 'phenotype']
snp_df = pd.read_csv('../data/UKBB_all_traits_all_sig_associations_CT20220207.bed', delimiter='\t', names=snp_cols)
print(snp_df)

trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])
print(trait_codes_df)

# TF data 
# NOTE: THIS IS CURRENTLY HG38
tf_pd = pd.read_pickle('../data/TF-df-hg38.pkl')
print(tf_pd)

# DHS data
# NOTE: THIS IS CURRENTLY HG38
dhs_df = pd.read_pickle('../data/dhs_metadata.pkl')
print(dhs_df)

dhs_Mixture_df = pd.read_pickle('../data/dhs_mixture_df.pkl')
print(dhs_Mixture_df)

dhs_enrichment_df = pd.read_pickle('../data/dhs_enrichment_df_25kb.pkl')
print(dhs_enrichment_df)

# Chromatin State Data
epilogos_833_df = pd.read_pickle('../data/epilogos_state_data_18.pkl')
print(epilogos_833_df)

epilogos_states_df = pd.read_pickle('../data/epilogos_states_df')
print(epilogos_states_df)

epilogos_metadata = pd.read_csv('../data/All_833_biosamples/metadata.tsv', delimiter='\t')
print(epilogos_metadata)


# Gene Data
string_assoc = pd.read_pickle('../data/hg19-protein-associations.pkl')
print(string_assoc)

gene_hg19_bed = pd.read_pickle('../data/hg19_gene_bed.pkl')
print(gene_hg19_bed)

protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
print(protein_assoc_hg19)

      seqname      start        end     ?
0        chr1       9000      34000  0.41
1        chr1      34000      59000  0.41
2        chr1      61000      86000  0.41
3        chr1      86000     111000  0.41
4        chr1     111000     136000  0.41
...       ...        ...        ...   ...
99991    chrX  155042800  155067800  0.41
99992    chrX  155067800  155092800  0.41
99993    chrX  155092800  155117800  0.41
99994    chrX  155117800  155142800  0.41
99995    chrX  155142800  155167800  0.41

[99996 rows x 4 columns]
         Chromosome         p1         p2 Reference SNP cluster ID  \
0             chr12  130300801  130300802               rs73150827   
1              chr4   99318162   99318163                rs1229984   
2              chr9    8303561    8303562              rs112120933   
3              chr1   46941503   46941504              rs141586057   
4             chr20   52932649   52932650              rs185961463   
...             ...        ...        ...         

identifier   1.10011   1.10021   1.10025   1.10027    1.1003   1.10032  \
0           0.001439  0.000000  0.000000  0.001336  0.000000  0.000000   
1           0.000000  0.000000  0.000000  0.001010  0.000000  0.000000   
2           0.000000  0.000000  0.000000  0.011659  0.000893  0.000000   
3           0.000000  0.000311  0.000000  0.008287  0.000000  0.000000   
4           0.000204  0.000000  0.000000  0.004569  0.000134  0.001297   
5           0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
6           0.000040  0.011431  0.000000  0.002904  0.006965  0.000000   
7           0.000000  0.000000  0.025745  0.001445  0.000000  0.000000   
8           0.000000  0.000000  0.000000  0.000000  0.000208  0.000000   
9           0.000000  0.000000  0.000000  0.000000  0.001768  0.000000   
10          0.000000  0.000000  0.000000  0.000000  0.003912  0.000000   
11          0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
12          0.000000  0.000000  0.0000

         seqname      start        end   0   1   2   3   4   5   6  ...  823  \
0           chr1          0        200  18  18  18  18  18  18  18  ...   18   
1           chr1        200        400  18  18  18  18  18  18  18  ...   18   
2           chr1        400        600  18  18  18  18  18  18  18  ...   18   
3           chr1        600        800  18  18  18  18  18  18  18  ...   18   
4           chr1        800       1000  18  18  18  18  18  18  18  ...   18   
...          ...        ...        ...  ..  ..  ..  ..  ..  ..  ..  ...  ...   
15181503    chrX  155269400  155269600  18  18  18  18  18  18  18  ...   18   
15181504    chrX  155269600  155269800  18  18  18  18  18  18  18  ...   18   
15181505    chrX  155269800  155270000  18  18  18  18  18  18  18  ...   18   
15181506    chrX  155270000  155270200  18  18  18  18  18  18  18  ...   18   
15181507    chrX  155270200  155270400  18  18  18  18  18  18  18  ...   18   

          824  825  826  827  828  829 

                                  protein2  neighborhood  fusion  cooccurence  \
protein1                                                                        
9606.ENSP00000000233  9606.ENSP00000379496           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000314067           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000263116           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000361263           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000409666           0.0     0.0          0.0   
...                                    ...           ...     ...          ...   
9606.ENSP00000469413  9606.ENSP00000397459           0.0     0.0          0.0   
9606.ENSP00000469413  9606.ENSP00000222399           0.0     0.0          0.0   
9606.ENSP00000469413  9606.ENSP00000215659           0.0     0.0        144.0   
9606.ENSP00000469413  9606.ENSP00000305918           0.0     0.0          0.0   
9606.ENSP00000469413  9606.E

In [ ]:
bins_25kb_bed = BedTool.from_dataframe(bins_25kb)
dhs_bed = BedTool.from_dataframe(dhs_df)

print(dhs_bed.map(bins_25kb_bed).to_dataframe())

In [6]:
print(bins_25kb_bed.to_dataframe())

      chrom      start        end  name
0      chr1       9000      34000  0.41
1      chr1      34000      59000  0.41
2      chr1      61000      86000  0.41
3      chr1      86000     111000  0.41
4      chr1     111000     136000  0.41
...     ...        ...        ...   ...
99991  chrX  155042800  155067800  0.41
99992  chrX  155067800  155092800  0.41
99993  chrX  155092800  155117800  0.41
99994  chrX  155117800  155142800  0.41
99995  chrX  155142800  155167800  0.41

[99996 rows x 4 columns]


In [7]:
print(dhs_bed.to_dataframe())

        chrom     start       end      name  score    strand  thickStart  \
0        chr1     16140     16200  0.129388      1     16170     16170.0   
1        chr1     51868     52040  0.080034      1     51970     51970.0   
2        chr1     57280     57354  0.273251      4     57350     57350.0   
3        chr1     66370     66482  0.183716      8     66430     66410.0   
4        chr1     79100     79231  0.113049      2     79150     79150.0   
...       ...       ...       ...       ...    ...       ...         ...   
3591893  chrY  56882540  56882719  0.038079      1  56882610  56882610.0   
3591894  chrY  56882864  56882980  0.115489      1  56882930  56882930.0   
3591895  chrY  56883733  56883960  0.491377      5  56883830  56883742.0   
3591896  chrY  56884440  56884580  0.053759      1  56884510  56884510.0   
3591897  chrY  56885400  56885520  0.062719      1  56885430  56885430.0   

           thickEnd                  itemRgb  
0           16170.0         Tissue invar

In [12]:
# dhs_bed
dhs_bed.saveas('../data/dhs_bed.bed')

<BedTool(../data/dhs_bed.bed)>

In [20]:
components = list(set(dhs_df.component.tolist()))
for c in components:
    dhs_sub = dhs_df.loc[dhs_df['component'] == c]
    dhs_sub_bed = BedTool.from_dataframe(dhs_sub)
    if ' / ' in c:
        c = c.replace(' / ', '_')
    if ' ' in c:
        c = c.replace(' ', '')
    if '.' in c:
        c = c.replace('.', '')
    print(c)
    dhs_sub_bed.saveas('../data/dhs_component_bed/' + c + '.bed')

Vascular_endothelial
Neural
StromalA
Tissueinvariant
Digestive
Placental_trophoblast
Lymphoid
StromalB
Myeloid_erythroid
Pulmonarydevel
Renal_cancer
Organdevel_renal
Cancer_epithelial
Cardiac
Primitive_embryonic
Musculoskeletal


In [78]:
def list_to_dict(str_list):
#     print(map(lambda s : map(str.strip, s.split(':')), str_list))
    return dict(map(lambda s : map(str.strip, s.split(':')), str_list))


counts_per_component_df = pd.read_csv('../src/results/counts_per_component.bed', delimiter='\t', header=None)
# # counts_per_component_df[4] = counts_per_component_df[4][0]
# print(counts_per_component_df[4][0])
counts_per_component_df[4] = counts_per_component_df[4].str.split(',')
counts_per_component_df[4] = counts_per_component_df[4].apply(list_to_dict)
dhs_counts_df = counts_per_component_df[4].apply(pd.Series)


dhs_counts_df = pd.concat([counts_per_component_df.iloc[:, :3], dhs_counts_df], axis=1)

In [79]:
dhs_counts_df = dhs_counts_df.rename({0:'seqname', 1:'start', 2:'end'}, axis=1)
dhs_counts_df

,seqname,start,end,Cancer_epithelial,Cardiac,Digestive,Lymphoid,Musculoskeletal,Myeloid_erythroid,Neural,Organdevel_renal,Placental_trophoblast,Primitive_embryonic,Pulmonarydevel,Renal_cancer,StromalA,StromalB,Tissueinvariant,Vascular_endothelial
0,chr1,9000,34000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,chr1,34000,59000,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0
2,chr1,61000,86000,0,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0
3,chr1,86000,111000,0,0,1,1,1,0,1,0,2,0,0,0,0,0,0,0
4,chr1,111000,136000,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,chrX,155042800,155067800,1,0,1,2,2,3,3,1,0,2,0,8,0,4,2,1
99992,chrX,155067800,155092800,2,2,2,2,1,4,1,2,2,3,0,0,0,2,1,2
99993,chrX,155092800,155117800,3,1,1,1,1,1,1,0,0,1,0,2,0,5,2,1
99994,chrX,155117800,155142800,1,0,1,0,1,4,1,0,1,4,0,2,0,1,0,1


In [81]:
dhs_counts_df.to_pickle('../data/dhs_counts_df_25kb.pkl')
dhs_counts_df = pd.read_pickle('../data/dhs_counts_df_25kb.pkl')
dhs_counts_df

,seqname,start,end,Cancer_epithelial,Cardiac,Digestive,Lymphoid,Musculoskeletal,Myeloid_erythroid,Neural,Organdevel_renal,Placental_trophoblast,Primitive_embryonic,Pulmonarydevel,Renal_cancer,StromalA,StromalB,Tissueinvariant,Vascular_endothelial
0,chr1,9000,34000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,chr1,34000,59000,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0
2,chr1,61000,86000,0,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0
3,chr1,86000,111000,0,0,1,1,1,0,1,0,2,0,0,0,0,0,0,0
4,chr1,111000,136000,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,chrX,155042800,155067800,1,0,1,2,2,3,3,1,0,2,0,8,0,4,2,1
99992,chrX,155067800,155092800,2,2,2,2,1,4,1,2,2,3,0,0,0,2,1,2
99993,chrX,155092800,155117800,3,1,1,1,1,1,1,0,0,1,0,2,0,5,2,1
99994,chrX,155117800,155142800,1,0,1,0,1,4,1,0,1,4,0,2,0,1,0,1


In [9]:
# tf_bed = BedTool.from_dataframe(tf_pd)
# tf_bed.saveas('../data/tf_bed.bed')

# tf_cols = ['chromosome', 'start', 'end', 'TF motif cluster/group', 'match score', 'strand', 'TF motif', '?']
# tf_fn = gzip.GzipFile('../data/moods.combined.all.bed.gz', "r")
# tf_pd = pd.read_csv(tf_fn, delimiter='\t', names=tf_cols)
# tf_pd.to_pickle('../data/TF-df-hg38.pkl')

KeyboardInterrupt: 

In [47]:
clusters = list(set(tf_pd['TF motif cluster/group']))
print(len(clusters))


# for c in clusters:
#     cluster_sub = tf_pd.loc[tf_pd['TF motif cluster/group'] == c]
#     cluster_sub_bed = BedTool.from_dataframe(cluster_sub)
#     if '/' in c:
#         c = c.replace('/', '_')
#     print(c)
#     cluster_sub_bed.saveas('../data/tf_motif_bed/' + c + '.bed')

282


In [48]:
print(len(tf_pd))

764808791


In [32]:
c_str = ""
for i, c in enumerate(clusters):
    if '/' in c:
        c = c.replace('/', '_')
    c_str = c_str + "\""+c+"\" "
print(c_str)

"ZNF324" "FOX_5" "ZFX" "ZBTB48" "NFKB_2" "EVI1_MECOM" "HAND1" "ZNF134" "BCL6_2" "ARI5A" "KLF_SP_1" "EGR" "GLI" "FEZF1" "HD_2" "ZNF384_1" "HD_21" "HOMEZ" "HD_8" "NR_19" "SREBF1" "ZIC" "ZNF257" "ZNF329" "ZNF382" "NR_11" "ZNF418" "HINFP1_1" "HD_14" "ZSCAN3" "NFI_3" "ZNF28" "HIC_2" "ZNF652" "NFAT_2" "FOX_9" "MZF1" "ZNF250" "ZBED1" "TBX_3" "INSM1" "SMAD" "ZNF136" "NR_16" "PRDM4" "CPEB1" "PAX_2" "HEN1" "NR_2" "FOX_2" "E2F_3" "SPI" "HD_18" "NFY" "SPDEF_1" "P53-like_1" "ZNF146" "ZNF332" "ZNF306" "PRDM1" "HD_3" "Ebox_CACGTG_2" "GATA" "ZNF41" "STAT_2" "YY1" "HD_16" "NR_5" "ZIM3" "GCM" "RFX_2" "ZKSCAN1" "RBPJ" "ZNF436" "HD_7" "RFX_1" "IRF_1" "HD_10" "EBF1" "GFI" "KLF_SP_3" "DMRT1" "MYB_5" "NFI_2" "ZNF384_2" "MAF" "HD_6" "IRF_4" "ZNF354" "FOX_7" "GMEB2_2" "HD_12" "SIX_1" "ZNF423" "CUX_3" "ZNF24" "HD_19" "ZIC_2" "ZNF547" "HD_23" "ZNF435" "ETS_1" "CUX_2" "AP1_1" "HD_5" "NR_12" "ZNF53" "TEAD" "TATA" "GC-tract" "GLIS" "SOX_5" "REST_NRSF" "PAX_1" "ZNF528" "ZSCAN4" "OVOL1" "SNAI2" "ZBTB6" "GMEB2_1" "NR_

In [ ]:
components = list(set(dhs_df.component.tolist()))
for c in components:
    dhs_sub = dhs_df.loc[dhs_df['component'] == c]
    dhs_sub_bed = BedTool.from_dataframe(dhs_sub)
    if ' / ' in c:
        c = c.replace(' / ', '_')
    if ' ' in c:
        c = c.replace(' ', '')
    if '.' in c:
        c = c.replace('.', '')
    print(c)
    dhs_sub_bed.saveas('../data/dhs_component_bed/' + c + '.bed')